# Day 1 · Exercise 2: SSML Prosody Control

**What you'll build:** `generate_audio_ssml` — a TTS function that accepts `rate` and `pitch` parameters and applies them using SSML markup.

**Why it matters:** Flat, one-speed TTS sounds robotic. Controlling rate and pitch lets you match the voice to the context — slower for learners, faster for real-time assistants. This is the pattern you'll use in every production TTS feature you build.

**How to complete this exercise:**
1. Read the docstring carefully — the signature and escaping requirement are specified there
2. Replace `pass` with your implementation
3. Run the **Check Your Work** cell — all 5 checks must pass

> **Hint if you're stuck:** The key is `html.escape(text)` before building the SSML string, and a `<prosody>` tag wrapping the escaped text.

## Your Implementation

In [ ]:
import edge_tts
import asyncio
import html

async def generate_audio_ssml(
    text: str,
    voice: str,
    output_path: str,
    rate: str = "+0%",
    pitch: str = "+0Hz",
) -> None:
    """
    Generate an MP3 using Edge TTS with SSML prosody control.

    Args:
        text:        The text to convert to speech.
        voice:       Edge TTS voice name, e.g. 'en-US-JennyNeural'.
        output_path: Where to save the MP3.
        rate:        Speech rate offset. '+0%' is neutral.
                     Examples: '+20%' (faster), '-20%' (slower).
        pitch:       Pitch offset in Hz. '+0Hz' is neutral.
                     Examples: '+10Hz' (higher), '-10Hz' (lower).

    Requirements:
        - Use html.escape() on the text before building the SSML string.
        - Wrap the escaped text in a <prosody> tag with rate and pitch attributes.
        - Pass the SSML string to edge_tts.Communicate as if it were plain text.

    Example:
        await generate_audio_ssml(
            text='Hello, world.',
            voice='en-US-JennyNeural',
            output_path='hello.mp3',
            rate='-20%',
        )
    """
    # ── YOUR CODE HERE ──────────────────────────────────────────────────────
    pass
    # ────────────────────────────────────────────────────────────────────────

## Check Your Work

Run the cell below. It runs 5 automated checks and reports exactly what passed.

In [ ]:
import os, asyncio

JENNY  = 'en-US-JennyNeural'
_PASS  = '\u2705'
_FAIL  = '\u274c'
_files = []

async def _run_checks():
    score = 0
    total = 5

    # Check 1: function is callable
    try:
        assert callable(generate_audio_ssml), 'generate_audio_ssml is not defined or not callable'
        print(f'{_PASS} Check 1/5: function exists and is callable')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 1/5: {e}')
        return

    # Check 2: creates a file at the specified path
    try:
        path = '__check_ssml_base.mp3'
        _files.append(path)
        await generate_audio_ssml('Testing one two three.', JENNY, path)
        assert os.path.exists(path), f'No file created at {path}'
        print(f'{_PASS} Check 2/5: creates a file at output_path')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 2/5: {e}')
        return

    # Check 3: file is a real audio file (> 2 KB)
    try:
        size = os.path.getsize(path)
        assert size > 2000, f'File is only {size} bytes — TTS call may have failed'
        print(f'{_PASS} Check 3/5: output is a real audio file ({size:,} bytes)')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 3/5: {e}')

    # Check 4: rate parameter changes the output
    # Faster rate → shorter audio → smaller MP3
    try:
        script = (
            'The quick brown fox jumped over the lazy dog. '
            'She sells sea shells by the sea shore. '
            'How much wood would a woodchuck chuck.'
        )
        path_fast = '__check_ssml_fast.mp3'
        path_slow = '__check_ssml_slow.mp3'
        _files.extend([path_fast, path_slow])

        await generate_audio_ssml(script, JENNY, path_fast, rate='+50%')
        await generate_audio_ssml(script, JENNY, path_slow, rate='-50%')

        size_fast = os.path.getsize(path_fast)
        size_slow = os.path.getsize(path_slow)

        assert size_fast < size_slow, (
            f'Fast file ({size_fast:,}B) is not smaller than slow file ({size_slow:,}B). '
            f'The rate parameter may not be applied — make sure you include it in the SSML tag.'
        )
        print(f'{_PASS} Check 4/5: rate parameter changes output '
              f'(fast={size_fast:,}B < slow={size_slow:,}B)')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 4/5: {e}')

    # Check 5: html.escape is applied (text with XML special chars doesn't crash)
    try:
        tricky = 'Price is <$5 & it\'s "great" today'
        path_escape = '__check_ssml_escape.mp3'
        _files.append(path_escape)
        await generate_audio_ssml(tricky, JENNY, path_escape)
        assert os.path.exists(path_escape) and os.path.getsize(path_escape) > 2000, \
            'Function crashed or produced empty output on text with XML special characters'
        print(f'{_PASS} Check 5/5: handles XML special characters safely (html.escape)')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 5/5: crashed on text with < > & characters — did you call html.escape()? Error: {e}')

    # Result
    print()
    if score == total:
        print('=' * 52)
        print(f'  {_PASS}  Exercise 2 complete! All {total}/{total} checks passed.')
        print('=' * 52)
    else:
        print(f'  {score}/{total} checks passed. Keep going — you\'re close!')

    for f in _files:
        if os.path.exists(f):
            os.remove(f)

await _run_checks()

## Bonus Challenge

Once all 5 checks pass, try this:

Use `generate_audio_ssml` to generate three versions of the same sentence with different rates — slow, normal, and fast. Listen to all three and notice where the voice starts to sound unnatural. What's the maximum rate increase before clarity breaks down?

This is a real calibration question you'd need to answer when building a reading tool.

In [ ]:
# Bonus: calibrate your rate limits
SENTENCE = "The quick brown fox jumped over the lazy dog."

# Uncomment and run each — listen to the results
# asyncio.run(generate_audio_ssml(SENTENCE, 'en-US-JennyNeural', 'rate_slow.mp3',   rate='-30%'))
# asyncio.run(generate_audio_ssml(SENTENCE, 'en-US-JennyNeural', 'rate_normal.mp3', rate='+0%'))
# asyncio.run(generate_audio_ssml(SENTENCE, 'en-US-JennyNeural', 'rate_fast.mp3',   rate='+30%'))
# asyncio.run(generate_audio_ssml(SENTENCE, 'en-US-JennyNeural', 'rate_vfast.mp3',  rate='+60%'))

---
## Solution

<details>
<summary>Click to reveal solution — try on your own first</summary>

```python
async def generate_audio_ssml(
    text: str,
    voice: str,
    output_path: str,
    rate: str = "+0%",
    pitch: str = "+0Hz",
) -> None:
    safe = html.escape(text)
    ssml = f'<prosody rate="{rate}" pitch="{pitch}">{safe}</prosody>'
    communicate = edge_tts.Communicate(ssml, voice)
    await communicate.save(output_path)
```

**Why this works:**
- `html.escape(text)` converts `<`, `>`, `&`, and `"` to their XML-safe equivalents (`&lt;`, `&gt;`, `&amp;`, `&quot;`). Without this, text containing those characters would break the SSML structure.
- The `<prosody>` tag is interpreted by Edge TTS before synthesis — it shifts how the voice engine produces audio, not the audio itself after the fact.
- Default values of `+0%` and `+0Hz` are neutral — they pass through to SSML but produce no change, making this a drop-in replacement for plain `generate_audio`.

**What the rate check is actually testing:**  
Faster speech means fewer samples over the same words → a shorter audio stream → a smaller MP3 file. This is a reliable side-effect test: you don't need to decode the audio to verify the rate was applied — file size tells you.

</details>